# 🚀 SimCSE: Unsupervised Pipeline, Ablations & Hugging Face Publishing
### **Google Colab Edition (Optimized for NVIDIA L4 GPU & High-RAM)**

**Author / Team Lead:** Russel Ku (Russ)  
**Team Responsibilities:** Unsupervised SimCSE pipeline, Unsupervised Ablation (*Same Dropout Mask*), Benchmark Evaluation, Hugging Face Hub Export & Verification.  
**Hardware Target:** NVIDIA L4 (24GB VRAM) / High-RAM Environment (53GB System RAM).  
**Paper Reference:** *SimCSE: Simple Contrastive Learning of Sentence Embeddings* (Gao et al., EMNLP 2021)  

---

## 📋 Workflow Outline
1. **⚡ Environment Setup & L4 Hardware Check:** Clone repository, install dependencies, enable CUDA & Tensor Cores.
2. **📊 Dataset Loading & Verification:** Process 100k SNLI dataset (~165k unique sentences) and STS-Benchmark (Dev/Test).
3. **🧪 Baseline Verification:** Evaluate Raw `bert-base-uncased` and `SBERT-2019` to verify STS-B harness.
4. **🎯 Unsupervised SimCSE Training:** Train `bert-base-uncased` with InfoNCE loss ($	au=0.05$) and independent dropout augmentation.
5. **🔬 Unsupervised Ablation Study:** Train with `same_dropout_mask=True` to empirically prove the collapse of contrastive learning without dropout noise.
6. **📈 Alignment & Uniformity Metrics:** Calculate Wang & Isola (2020) representation metrics.
7. **🤗 Hugging Face Hub Export & Verification:** Convert model to `SentenceTransformers`, generate Model Card, and verify reloading from the Hub.

## 1. ⚙️ Environment Setup & Repository Clone

In [ ]:
# 1. Clone repository (or pull latest updates)
import os
if not os.path.exists("Sentence-embedding-model"):
    !git clone https://github.com/RusselKu/Sentence-embedding-model.git
    %cd Sentence-embedding-model
else:
    %cd Sentence-embedding-model
    !git pull

# 2. Install required packages (Fast)
!pip install -q --upgrade transformers datasets sentence-transformers scikit-learn scipy tqdm huggingface_hub

# 3. Verify CUDA & L4 GPU
import torch
print("=" * 60)
print(f"[✓] PyTorch Version: {torch.__version__}")
print(f"[✓] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[✓] Device Name: {torch.cuda.get_device_name(0)}")
    print(f"[✓] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"[✓] CUDA Capability: {torch.cuda.get_device_capability(0)}")
    # Enable TF32 for NVIDIA Ada Lovelace / L4 Tensor Cores speedup
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("[✓] TF32 Tensor Core Acceleration: ENABLED")
print("=" * 60)

## 2. 📂 Ensure Dataset & Verify STS-Benchmark
Let's ensure `data/snli_train_100k.jsonl` is present and inspect the STS-B Dev/Test validation sets.

In [ ]:
import os
import sys
sys.path.insert(0, os.getcwd())

from src.data import build_unsupervised_dataset, load_stsb_dataset

# Ensure data path
data_path = "data/snli_train_100k.jsonl"
if not os.path.exists(data_path) and os.path.exists("snli_train_100k.jsonl"):
    os.makedirs("data", exist_ok=True)
    !cp snli_train_100k.jsonl data/snli_train_100k.jsonl

# Load and verify unsupervised sentences
unique_sentences = build_unsupervised_dataset(data_path)
print(f"[✓] Extracted {len(unique_sentences):,} unique sentences from SNLI 100k.")

# Load STS-B
dev_records, test_records = load_stsb_dataset()
print(f"[✓] Loaded STS-B Benchmark: {len(dev_records)} Dev pairs, {len(test_records)} Test pairs.")
print(f"Sample STS-B pair: '{dev_records[0]['sentence1']}' <-> '{dev_records[0]['sentence2']}' (Score: {dev_records[0]['score']})")

## 3. 🧪 Baseline Sanity Checks
Verify the STS-B evaluation harness against established reference scores:
- **Raw `bert-base-uncased` (Mean Pooling):** Expected Dev $\approx 59.31$ / Test $\approx 47.29$
- **SBERT-2019 (`bert-base-nli-mean-tokens`):** Expected Dev $\approx 80.77$ / Test $\approx 76.98$

In [ ]:
!python scripts/run_sanity_checks.py

## 4. 🎯 Run 1: Unsupervised SimCSE Training (Default Baseline)
- **Backbone:** `bert-base-uncased` (110M params)
- **Batch Size:** $64$
- **Learning Rate:** $3\times 10^{-5}$
- **Epochs:** $1$
- **Temperature ($\tau$):** $0.05$
- **Pooling:** `cls` token with MLP projection head during training
- **Data Augmentation:** Independent Dropout ($p=0.1$)

In [ ]:
!python scripts/train_unsupervised.py \
  --data_path data/snli_train_100k.jsonl \
  --lr 3e-5 \
  --batch_size 64 \
  --epochs 1 \
  --temperature 0.05 \
  --pooling cls \
  --seed 42 \
  --run_name unsup_simcse_default

## 5. 🔬 Run 2: Unsupervised Ablation Study (*Same Dropout Mask*)

**Ablation Hypothesis:**
If we pass the sentence through the encoder twice with the **exact same dropout mask** (`same_dropout_mask=True`), both views become identical ($z_1 = z_2$). Without representation variance, the InfoNCE loss collapses towards trivial solutions and semantic quality drops.

In [ ]:
!python scripts/run_ablations.py \
  --ablation unsup \
  --data_path data/snli_train_100k.jsonl

## 6. 📊 Comparative Metrics & Ablation Results Table

In [ ]:
import json

with open("runs/unsup_simcse_default.json", "r") as f:
    unsup_res = json.load(f)
with open("runs/ablation_unsup_same_mask.json", "r") as f:
    ablation_res = json.load(f)

u_dev = unsup_res['results']['dev']['spearman']
u_test = unsup_res['results']['test']['spearman']
u_align = unsup_res['results']['test']['alignment']
u_unif = unsup_res['results']['test']['uniformity']

a_dev = ablation_res['results']['dev']['spearman']
a_test = ablation_res['results']['test']['spearman']
a_align = ablation_res['results']['test']['alignment']
a_unif = ablation_res['results']['test']['uniformity']

print("=" * 95)
print(f"{'Experiment Model':<35} | {'Dev Spearman':<14} | {'Test Spearman':<14} | {'Alignment (α=2)':<16} | {'Uniformity (t=2)'}")
print("=" * 95)
print(f"{'Unsup Default (Independent Dropout)':<35} | {u_dev:<14.2f} | {u_test:<14.2f} | {u_align:<16.4f} | {u_unif:.4f}")
print(f"{'Unsup Ablation (Same Dropout Mask)':<35} | {a_dev:<14.2f} | {a_test:<14.2f} | {a_align:<16.4f} | {a_unif:.4f}")
print("-" * 95)
print(f"[Δ Impact]: Independent Dropout yields +{u_dev - a_dev:.2f} Dev and +{u_test - a_test:.2f} Test Spearman points.")
print("=" * 95)

## 7. 🤗 Export & Publish to Hugging Face Hub (with Post-Upload Verification)

This step:
1. Converts the best PyTorch checkpoint into a native `SentenceTransformers` model.
2. Auto-generates a rich **Model Card (`README.md`)** with STS-B evaluation metrics and pipeline tags.
3. Uploads the model to Hugging Face Hub.
4. **Verification Step:** Reloads the model live from the Hub (`SentenceTransformer('<your-repo-id>')`) and validates STS-B Test score match.

In [ ]:
from huggingface_hub import notebook_login

# 1. Login to Hugging Face (Paste your write token)
notebook_login()

In [ ]:
# Set your desired Hugging Face repository ID (e.g., 'RusselKu/simcse-bert-uncased-unsup')
HF_REPO_ID = "RusselKu/simcse-bert-uncased-unsup"  # <-- Cambia esto si deseas otro repo_id

!python src/publish.py \
  --checkpoint_dir checkpoints/unsup_simcse_default/best_checkpoint \
  --repo_id {HF_REPO_ID} \
  --run_json runs/unsup_simcse_default.json \
  --pooling cls

## 8. 🧪 Live Test: Semantic Similarity Demo

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load your published model
model = SentenceTransformer(HF_REPO_ID)

sentences = [
    "A dog is running across the grass.",
    "A puppy is playing on the lawn outside.",
    "A chef is preparing a delicious Italian meal in the kitchen."
]

embeddings = model.encode(sentences, normalize_embeddings=True)
sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix:")
print(f"- 'Dog running' vs 'Puppy playing': {sim_matrix[0, 1]:.4f} (High semantic match)")
print(f"- 'Dog running' vs 'Chef cooking':   {sim_matrix[0, 2]:.4f} (Low semantic match)")

## 9. 💾 Push Artifacts & Sync Run Logs back to GitHub

In [ ]:
# Check generated run logs and artifacts
!ls -la runs/
!ls -la checkpoints/